# Food Waste Prediction - Data Preprocessing

This notebook performs data cleaning, feature engineering, and preprocessing for the Food Waste Prediction System.

### Objectives
- Load the training dataset
- Clean and standardize the data
- Handle missing values
- Perform feature engineering
- Separate features and target
- Split the data into training and testing sets
- Build a preprocessing pipeline for numerical and categorical features

## 1. Import Libraries and Load Dataset

Import the required libraries and load the training dataset from the project dataset directory.

In [11]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../dataset/train.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (911, 12)


,ID,date,meals_served,kitchen_staff,temperature_C,humidity_percent,day_of_week,special_event,past_waste_kg,staff_experience,waste_category,food_waste_kg
0,0,2022-12-19,196,13,27.887273,45.362854,0,0,7.740587,intermediate,dairy,28.946465
1,1,2023-11-21,244,15,10.317872,64.430475,1,0,42.311779,NaN,MeAt,51.549053
2,4,2022-02-01,148,16,27.714300,69.046113,1,0,41.184305,Beginner,MeAt,53.008323
3,5,2023-03-19,157,19,19.173902,46.292823,6,0,41.543492,Beginner,MeAt,48.621527
4,6,2022-07-18,297,10,26.375233,79.741064,0,0,26.525097,Intermediate,MEAT,44.156984


## 2. Initial Data Inspection

Inspect the dataset structure, data types, missing values, and basic information before applying preprocessing steps.

In [12]:
df.info()

print("\nMissing values:")
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 911 entries, 0 to 910
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                911 non-null    int64  
 1   date              911 non-null    object 
 2   meals_served      911 non-null    int64  
 3   kitchen_staff     911 non-null    int64  
 4   temperature_C     911 non-null    float64
 5   humidity_percent  911 non-null    float64
 6   day_of_week       911 non-null    int64  
 7   special_event     911 non-null    int64  
 8   past_waste_kg     911 non-null    float64
 9   staff_experience  747 non-null    object 
 10  waste_category    911 non-null    object 
 11  food_waste_kg     911 non-null    float64
dtypes: float64(4), int64(5), object(3)
memory usage: 85.5+ KB

Missing values:
ID                    0
date                  0
meals_served          0
kitchen_staff         0
temperature_C         0
humidity_percent      0
day_of_week  

## 3. Data Cleaning

The dataset is cleaned by removing the identifier column and standardizing categorical values. Missing values are retained at this stage and will be handled automatically within the preprocessing pipeline.

In [13]:
# Remove identifier
df = df.drop(columns=["ID"])

# Standardize categorical values
df["staff_experience"] = (
    df["staff_experience"]
    .astype("object")
    .str.strip()
    .str.lower()
)

df["waste_category"] = (
    df["waste_category"]
    .astype("object")
    .str.strip()
    .str.lower()
)

## 4. Feature Engineering

New features are created from the existing data to provide the model with additional information that may help predict food waste.

The engineered features include:
- Year, month, and day from the date
- Weekend indicator
- Meals served per kitchen staff member

In [14]:
# Convert date
df["date"] = pd.to_datetime(df["date"])

# Date features
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day_of_month"] = df["date"].dt.day

# Weekend indicator
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

# Workload feature
df["meals_per_staff"] = (
    df["meals_served"] /
    df["kitchen_staff"].replace(0, np.nan)
)

# Remove original date
df = df.drop(columns=["date"])

df.head()

,meals_served,kitchen_staff,temperature_C,humidity_percent,day_of_week,special_event,past_waste_kg,staff_experience,waste_category,food_waste_kg,year,month,day_of_month,is_weekend,meals_per_staff
0,196,13,27.887273,45.362854,0,0,7.740587,intermediate,dairy,28.946465,2022,12,19,0,15.076923
1,244,15,10.317872,64.430475,1,0,42.311779,NaN,meat,51.549053,2023,11,21,0,16.266667
2,148,16,27.714300,69.046113,1,0,41.184305,beginner,meat,53.008323,2022,2,1,0,9.250000
3,157,19,19.173902,46.292823,6,0,41.543492,beginner,meat,48.621527,2023,3,19,1,8.263158
4,297,10,26.375233,79.741064,0,0,26.525097,intermediate,meat,44.156984,2022,7,18,0,29.700000


## 5. Separate Features and Target

The target variable is `food_waste_kg`, which represents the amount of food waste in kilograms. The remaining columns are used as input features.

In [15]:
X = df.drop(columns=["food_waste_kg"])
y = df["food_waste_kg"]

print("Features:", X.shape)
print("Target:", y.shape)

Features: (911, 14)
Target: (911,)


## 6. Train-Test Split

The dataset is divided into training and testing sets. 80% of the data is used for training and 20% is reserved for evaluating the model on unseen data.

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

Training set: (728, 14)
Testing set: (183, 14)


## 7. Build the Preprocessing Pipeline

Numerical and categorical features require different preprocessing techniques.

For numerical features, missing values are replaced using the median and values are standardized.

For categorical features, missing values are replaced using the most frequent category and categorical values are converted using one-hot encoding.

The `ColumnTransformer` combines both preprocessing workflows into a single reusable pipeline.

In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = [
    "meals_served",
    "kitchen_staff",
    "temperature_C",
    "humidity_percent",
    "day_of_week",
    "special_event",
    "past_waste_kg",
    "year",
    "month",
    "day_of_month",
    "is_weekend",
    "meals_per_staff"
]

categorical_features = [
    "staff_experience",
    "waste_category"
]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

## 8. Verify the Preprocessing Pipeline

The preprocessing pipeline is fitted using the training data and applied to both training and testing sets. This ensures that preprocessing is learned only from the training data and can be consistently reused during model prediction.

In [18]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training data shape:", X_train_processed.shape)
print("Processed testing data shape:", X_test_processed.shape)

Processed training data shape: (728, 19)
Processed testing data shape: (183, 19)
